# Reconhecimento de Objetos com OpenCV e MediaPipe

Este notebook demonstra como realizar o reconhecimento de objetos em tempo real usando a webcam, OpenCV e as APIs de Tarefas do MediaPipe da Google.

In [ ]:
# Já instalei as dependências via 'uv' para você.
# Se precisar instalar manualmente no futuro, use:
# !uv pip install opencv-python mediapipe numpy requests

In [5]:
import cv2
import mediapipe as mp
import numpy as np
import os
import requests
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# Configurações do Modelo
MODEL_PATH = "efficientdet_lite0.tflite"
URL = "https://storage.googleapis.com/mediapipe-models/object_detector/efficientdet_lite0/float16/1/efficientdet_lite0.tflite"

# Download do modelo se não existir
if not os.path.exists(MODEL_PATH):
    print(f"Fazendo download do modelo {MODEL_PATH}...")
    r = requests.get(URL, allow_redirects=True)
    open(MODEL_PATH, 'wb').write(r.content)
    print("Download concluído!")

In [6]:
def draw_landmarks_on_image(rgb_image, detection_result):
    """Desenha as caixas delimitadoras e labels no frame."""
    for detection in detection_result.detections:
        # Coordenadas da Bounding Box
        bbox = detection.bounding_box
        start_point = int(bbox.origin_x), int(bbox.origin_y)
        end_point = int(bbox.origin_x + bbox.width), int(bbox.origin_y + bbox.height)
        cv2.rectangle(rgb_image, start_point, end_point, (0, 255, 0), 3)

        # Label e Score
        category = detection.categories[0]
        category_name = category.category_name
        probability = round(category.score, 2)
        result_text = f"{category_name} ({probability})"
        text_location = (int(bbox.origin_x) + 10, int(bbox.origin_y) + 25)
        cv2.putText(rgb_image, result_text, text_location, cv2.FONT_HERSHEY_PLAIN,
                    2, (0, 255, 0), 2)

    return rgb_image

# Inicializar o Detector
base_options = python.BaseOptions(model_asset_path=MODEL_PATH)
options = vision.ObjectDetectorOptions(base_options=base_options,
                                        score_threshold=0.5)
detector = vision.ObjectDetector.create_from_options(options)

# Abrir Webcam
cap = cv2.VideoCapture(0)

# Verificar se a webcam foi aberta com sucesso
if not cap.isOpened():
    print("ERRO: Não foi possível acessar a webcam. Verifique se ela está conectada.")
else:
    print("Webcam acessada com sucesso. Iniciando detecção...")
    print("Pressione 'q' na janela de vídeo para sair.")

    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            print("Erro ao ler o frame da webcam. Finalizando...")
            break

        # O MediaPipe requer imagem em RGB
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)

        # Detectar objetos no frame
        detection_result = detector.detect(mp_image)

        # Desenhar resultados
        annotated_image = draw_landmarks_on_image(frame, detection_result)

        # Mostrar o frame
        cv2.imshow('MediaPipe Object Detection', annotated_image)

        # Pressionar 'q' para sair
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()
    print("Webcam encerrada.")

ERRO: Não foi possível acessar a webcam. Verifique se ela está conectada.
